# Imports and seed setting

In [0]:
%pip install torch matplotlib scikit-learn shap 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
from __future__ import annotations

In [0]:
import gc
import logging
import math
import os
import pickle
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

In [0]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for server environments
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

/local_disk0/.ephemeral_nfs/envs/pythonEnv-1b683620-18c4-498d-b505-f570cbb06893/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
import vitaldb

In [0]:
# ── Reproducibility ──────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# CONFIG dict (ALL hyperparameters and settings)

In [0]:
CONFIG: Dict[str, Any] = {
    # ── Identity ──────────────────────────────────────────────────────────────
    "model_id": 5,
    "model_name": "PatientContextMLP",
    "output_file": "model_5.py",
    "log_file": "model_5.log",
    "checkpoint_path": "model_5_best.pt",
    "shap_output": "shap_model_5.png",
    "pred_output": "prediction_model_5.png",
    # ── VitalDB data range ────────────────────────────────────────────────────
    "caseid_min": 1,
    "caseid_max": 6388,
    # ── Clinical metadata fields to load ─────────────────────────────────────

    # Model 5 loads ONLY clinical metadata + labs (no waveform tracks)
    "clinical_params": [
        "caseid", "age", "sex", "bmi", "asa", "emop",
        "ane_type", "optype", "department", "position",
        "preop_htn", "preop_dm",
        "anestart", "aneend", "opstart", "opend",
    ],
    "lab_params": ["alb", "hb", "gluc"],
    # ── Feature column definitions ────────────────────────────────────────────
    "continuous_features": ["age", "bmi", "preop_alb", "preop_hb", "preop_gluc"],
    "ordinal_features": ["asa"],                      # kept as-is numeric
    "binary_features": ["sex", "emop", "preop_htn", "preop_dm"],
    "onehot_features": ["ane_type", "optype", "department", "position"],
    # ── Emergence timing label ────────────────────────────────────────────────
    # emergence_sec = aneend - opend; keep only 0 < value < 3600 s
    "et_min_sec": 0.0,
    "et_max_sec": 3600.0,
    # ── MLP architecture ──────────────────────────────────────────────────────
    "embedding_dim": 128,           # final output embedding dimension
    "hidden_dims": [64, 128, 64],   # hidden layer sizes (before final 128 output)
    "dropout": 0.3,
    # ─ Training ──────────────────────────────────────────────────────────────
    "batch_size": 256,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 100,
    "early_stopping_patience": 10,
    "grad_clip_norm": 1.0,
    # ── Data split ────────────────────────────────────────────────────────────
    "val_frac": 0.10,
    "test_frac": 0.10,
    "random_seed": 42,
    # ── Missing value strategy ────────────────────────────────────────────────
    # labs filled with training-set median; categoricals with training-set mode
    "lab_missing_strategy": "median",
    "cat_missing_strategy": "mode",
    # ── Minimum case duration ─────────────────────────────────────────────────
    "min_case_duration_sec": 1800,  # 30 min
}

# Logging setup

In [0]:
def setup_logging(log_file: str) -> logging.Logger:
    """
    Configure root logger with both a file handler and a console handler.
    Parameters
    ---------
    log_file : str
        Path to the log file.
    Returns
    ------
    logging.Logger
        Configured logger instance.
    """
    logger = logging.getLogger("model_5")
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    # File handler — full DEBUG output
    fh = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    # Console handler — INFO and above
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    return logger
LOGGER = setup_logging(CONFIG["log_file"])

#  load_clinical_metadata()

In [0]:
def load_clinical_metadata(config: Dict[str, Any]) -> pd.DataFrame:
    """
    Load per-case clinical metadata from VitalDB.
    Uses vitaldb.load_clinical_data() which returns a pandas DataFrame.
    All time columns (anestart, aneend, opstart, opend) are seconds from
    casestart (casestart = 0 for de-identification); negative values are valid.
    Parameters
    ---------
    config : dict
        CONFIG dict containing caseid_min, caseid_max, clinical_params.
    Returns
    ------
    pd.DataFrame
        Clinical metadata with one row per case.
    """
    all_caseids = list(range(config["caseid_min"], config["caseid_max"] + 1))
    LOGGER.info("Loading clinical metadata for %d cases …", len(all_caseids))
    try:
        df = vitaldb.load_clinical_data(
            caseids=all_caseids,
            params=config["clinical_params"],
        )
    except Exception as exc:
        LOGGER.error("load_clinical_data() failed: %s", exc)
        raise
    LOGGER.info("Clinical metadata loaded: %d rows, %d cols", len(df), len(df.columns))
    return df

# load_lab_data_wide()

In [0]:
def load_lab_data_wide(config: Dict[str, Any]) -> pd.DataFrame:
    """
    Load pre-operative laboratory values from VitalDB and pivot to wide format.
    vitaldb.load_lab_data() returns a long-format DataFrame with columns:
        caseid, dt (seconds from casestart), name, result
    We take the earliest (most pre-operative) measurement of each lab per case
    and pivot to wide format: columns = caseid + one column per lab name.
    Resulting column names are prefixed with 'preop_' to match CONFIG.
    Parameters
    ---------
    config : dict
        CONFIG dict containing caseid_min, caseid_max, lab_params.
    Returns
    ------
    pd.DataFrame
        Wide-format lab data, one row per caseid.
    """
    all_caseids = list(range(config["caseid_min"], config["caseid_max"] + 1))
    LOGGER.info("Loading lab data for %d cases …", len(all_caseids))
    try:
        df_long = vitaldb.load_lab_data(
            caseids=all_caseids,
            params=config["lab_params"],
        )
    except Exception as exc:
        LOGGER.error("load_lab_data() failed: %s", exc)
        raise
    if df_long is None or len(df_long) == 0:
        LOGGER.warning("load_lab_data() returned empty DataFrame")
        return pd.DataFrame({"caseid": all_caseids})
    # Take the earliest pre-operative value (smallest dt) for each caseid × lab
    df_long = df_long.dropna(subset=["result"])
    df_long["result"] = pd.to_numeric(df_long["result"], errors="coerce")
    df_preop = (
        df_long
        .sort_values("dt")
        .groupby(["caseid", "name"], as_index=False)
        .first()
    )
    # Pivot to wide — one column per lab
    df_wide = df_preop.pivot_table(
        index="caseid", columns="name", values="result", aggfunc="first"
    ).reset_index()
    # Rename columns to match expected feature names: alb → preop_alb, etc.
    df_wide.columns = [
        "caseid" if c == "caseid" else f"preop_{c}"
        for c in df_wide.columns
    ]
    LOGGER.info("Lab data wide shape: %s", df_wide.shape)
    return df_wide

# build_et_labels()

In [0]:
def build_et_labels(df: pd.DataFrame, config: Dict[str, Any]) -> pd.DataFrame:
    """
    Compute emergence timing label: emergence_sec = aneend - opend.
    Filters to cases where 0 < emergence_sec < 3600 s (CONFIG et_min_sec /
    et_max_sec). Rows outside this range are kept but labelled NaN so they
    can be excluded from loss computation without dropping from the dataset.
    The label is stored in log-space (log_emergence_sec) for log-normal MSE
    regression, matching Model 2's pretraining convention.
    Parameters
    ---------
    df : pd.DataFrame
        Merged clinical DataFrame containing aneend and opend columns.
    config : dict
        CONFIG dict.
    Returns
    ------
    pd.DataFrame
        Input DataFrame with two new columns:
        - emergence_sec    : raw seconds (float, NaN if out-of-range)
        - log_emergence_sec: log(emergence_sec) for regression target
    """
    df = df.copy()
    # Compute raw emergence duration
    df["emergence_sec"] = pd.to_numeric(df["aneend"], errors="coerce") - \
                          pd.to_numeric(df["opend"],  errors="coerce")
    # Mask out-of-range values — they will be excluded from loss but not dropped
    valid_mask = (
        df["emergence_sec"].notna() &
        (df["emergence_sec"] > config["et_min_sec"]) &
        (df["emergence_sec"] < config["et_max_sec"])
    )
    df.loc[~valid_mask, "emergence_sec"] = np.nan
    # Log-transform for log-normal regression (log1p for numerical safety)
    df["log_emergence_sec"] = np.log(df["emergence_sec"])  # NaN propagates
    n_valid = valid_mask.sum()
    LOGGER.info(
        "ET labels: %d / %d cases have valid emergence_sec in (%.0f, %.0f) s",
        n_valid, len(df), config["et_min_sec"], config["et_max_sec"]
    )
    return df

# SECTION 5b — remove_outliers() for continuous features

In [0]:
# Physiological plausibility bounds for patient static features
_PHYS_BOUNDS: Dict[str, Tuple[float, float]] = {
    "age":        (0.0,   120.0),
    "bmi":        (10.0,  80.0),
    "preop_alb":  (1.0,   6.0),    # g/dL
    "preop_hb":   (3.0,   22.0),   # g/dL
    "preop_gluc": (30.0,  600.0),  # mg/dL
}
def remove_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clip continuous features to physiological plausibility bounds.
    Values outside bounds are set to NaN (not clipped) so they are treated
    as missing and filled by the training-set median, preserving statistical
    integrity rather than silently biasing toward the boundary.
    Parameters
    ---------
    df : pd.DataFrame
        Patient-level DataFrame containing continuous feature columns.
    Returns
    ------
    pd.DataFrame
        DataFrame with out-of-range values replaced by NaN.
    """
    df = df.copy()
    for col, (lo, hi) in _PHYS_BOUNDS.items():
        if col not in df.columns:
            continue
        n_before = df[col].notna().sum()
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
        n_after = df[col].notna().sum()
        if n_before != n_after:
            LOGGER.debug(
                "Outlier removal: %s — set %d values to NaN",
                col, n_before - n_after
            )
    return df

# Feature engineering: one-hot encoding + imputation helpers

In [0]:
def build_feature_matrix(
    df: pd.DataFrame,
    config: Dict[str, Any],
    fit_mode: bool,
    artifacts: Optional[Dict[str, Any]] = None,
) -> Tuple[np.ndarray, List[str], Dict[str, Any]]:
    """
    Construct the numeric feature matrix from the merged patient DataFrame.
    Processing pipeline:
    1. Continuous features  → StandardScaler (fitted on train only)
    2. Ordinal features     
→ kept as-is numeric
    3. Binary features      
    4. One-hot features     
    5. Missing imputation   
→ kept as-is (0/1)
→ pd.get_dummies with sorted categories
→ median (continuous/labs) or mode (categorical)
    Parameters
    ---------
    df : pd.DataFrame
        Merged patient-level DataFrame (clinical + labs + ET label).
    config : dict
        CONFIG dict.
    fit_mode : bool
        If True, fit scalers/imputers on this data (training split).
        If False, apply pre-fitted artifacts from `artifacts`.
    artifacts : dict or None
        Pre-fitted preprocessing objects (required when fit_mode=False).
    Returns
    ------
    X : np.ndarray
        Feature matrix of shape (N, n_features), dtype float32.
    feature_names : list of str
        Ordered list of feature names corresponding to columns of X.
    artifacts : dict
        Fitted preprocessing objects (scaler, medians, modes, onehot_cols).
    """
    df = df.copy()
    if artifacts is None:
        artifacts = {}
    # ── Step 1: Impute continuous + lab features ──────────────────────────────
    for col in config["continuous_features"]:
        if col not in df.columns:
            df[col] = np.nan
        if fit_mode:
            # Compute and store training-set median
            median_val = float(df[col].median()) if df[col].notna().any() else 0.0
            artifacts[f"median_{col}"] = median_val
        else:
            median_val = artifacts[f"median_{col}"]
        df[col] = df[col].fillna(median_val)
    # ── Step 2: Impute and keep ordinal features (asa: 1–5) ──────────────────
    for col in config["ordinal_features"]:
        if col not in df.columns:
            df[col] = np.nan
        if fit_mode:
            mode_val = float(df[col].mode().iloc[0]) if df[col].notna().any() else 2.0
            artifacts[f"mode_{col}"] = mode_val
        else:
            mode_val = artifacts[f"mode_{col}"]
        df[col] = df[col].fillna(mode_val).astype(float)
    # ── Step 3: Impute binary features ────────────────────────────────────────
    for col in config["binary_features"]:
        if col not in df.columns:
            df[col] = 0.0
        if fit_mode:
            mode_val = float(df[col].mode().iloc[0]) if df[col].notna().any() else 0.0
            artifacts[f"mode_{col}"] = mode_val
        else:
            mode_val = artifacts[f"mode_{col}"]
     
        df[col] = df[col].fillna(mode_val).astype(float)

    # ── Step 4: One-hot encode categorical features ───────────────────────────

    onehot_frames: List[pd.DataFrame] = []
    for col in config["onehot_features"]:
        if col not in df.columns:
            df[col] = "unknown"
        df[col] = df[col].fillna("unknown").astype(str)
        if fit_mode:
            # Determine and store all categories seen in training
            cats = sorted(df[col].unique().tolist())
            artifacts[f"onehot_cats_{col}"] = cats
        else:
            cats = artifacts[f"onehot_cats_{col}"]
        # Encode using training-determined categories; unseen → all zeros
        dummies = pd.get_dummies(df[col], prefix=col)
        expected_cols = [f"{col}_{c}" for c in cats]
        for ec in expected_cols:
            if ec not in dummies.columns:
                dummies[ec] = 0
        dummies = dummies[expected_cols]  # enforce consistent column order
        onehot_frames.append(dummies.reset_index(drop=True))

    # ── Step 5: Assemble feature columns ─────────────────────────────────────
    scalar_cols = (
        config["continuous_features"] +
        config["ordinal_features"] +
        config["binary_features"]
    )
    df_reset = df[scalar_cols].reset_index(drop=True).astype(float)
    all_frames = [df_reset] + onehot_frames
    df_combined = pd.concat(all_frames, axis=1)
    feature_names: List[str] = list(df_combined.columns)

    # ── Step 6: Scale continuous features with StandardScaler ─────────────────
    cont_indices = [feature_names.index(c) for c in config["continuous_features"]]
    X = df_combined.values.astype(np.float32)
    if fit_mode:
        scaler = StandardScaler()
        X[:, cont_indices] = scaler.fit_transform(X[:, cont_indices])
        artifacts["scaler"] = scaler
        artifacts["feature_names"] = feature_names
        LOGGER.info(
            "Fitted scaler on %d training samples; %d features total",
            len(X), len(feature_names)
        )
    else:
        scaler = artifacts["scaler"]
        X[:, cont_indices] = scaler.transform(X[:, cont_indices])
        
    # Replace any residual NaN (from edge cases) with 0.0
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        LOGGER.warning("Replacing %d residual NaN values in feature matrix with 0", nan_count)
        X = np.nan_to_num(X, nan=0.0)
    return X, feature_names, artifacts

# SECTION 7 (data split portion) — split_cases()

In [0]:
def split_cases(
    all_caseids: List[int],
    val_frac: float = 0.10,
    test_frac: float = 0.10,
    seed: int = 42,
) -> Tuple[List[int], List[int], List[int]]:
    """
    Patient-level train / validation / test split.
    Never splits within a patient across sets. Each caseid is a unique patient
    in VitalDB, so a random split is appropriate.
    Parameters
    ---------
    all_caseids : list of int
        All eligible case IDs.
    val_frac : float
        Fraction of total data for validation.
    test_frac : float
        Fraction of total data for test.
    seed : int
        Random seed for reproducibility.
    Returns
    ------
    train, val, test : tuple of lists
        Case IDs for each split.
    """
    train_val, test = train_test_split(
        all_caseids, test_size=test_frac, random_state=seed
    )
    # Adjust val fraction relative to the train+val subset
    adjusted_val_frac = val_frac / (1.0 - test_frac)
    train, val = train_test_split(
        train_val, test_size=adjusted_val_frac, random_state=seed
    )
    LOGGER.info(
        "Split: train=%d, val=%d, test=%d", len(train), len(val), len(test)
    )
    return train, val, test

# SECTION 8 — apply_eligibility_filter()

In [0]:
def apply_eligibility_filter(
    df: pd.DataFrame,
    config: Dict[str, Any],
) -> pd.DataFrame:
    """
    Filter cases to those meeting minimum quality criteria.
    Criteria applied:
    - Case duration (aneend - anestart) >= min_case_duration_sec
    - caseid is not NaN
    Note: Model 5 does not load waveforms, so SQI-based filters do not apply.
    The ET label filter (0 < emergence_sec < 3600) is handled separately in
    build_et_labels(); cases with NaN ET label are kept but excluded from loss.
    Parameters
    ---------
    df : pd.DataFrame
        Merged clinical DataFrame.
    config : dict
        CONFIG dict.
    Returns
    ------
    pd.DataFrame
        Filtered DataFrame.
    """
    n_before = len(df)
    df = df.dropna(subset=["caseid"]).copy()
    df["caseid"] = df["caseid"].astype(int)
    # Duration filter
    df["_case_duration"] = (
        pd.to_numeric(df["aneend"],   errors="coerce") 
        pd.to_numeric(df["anestart"], errors="coerce")
    )
    df = df[df["_case_duration"] >= config["min_case_duration_sec"]].copy()
    df = df.drop(columns=["_case_duration"])
    LOGGER.info(
        "Eligibility filter: %d → %d cases (removed %d)",
        n_before, len(df), n_before - len(df)
    )
    return df

# SECTION 9 — Model5Dataset (PyTorch Dataset)

In [0]:
class Model5Dataset(Dataset):
    """
    PyTorch Dataset for the Patient Context MLP (Model 5).
    Each item is a single patient represented by a static feature vector.
    Because Model 5 outputs a single embedding per patient (not per timestep),
    there is one item per case in the dataset.
    Parameters
    ---------
    X : np.ndarray
        Feature matrix of shape (N, n_features).
    caseids : list of int
        Case IDs corresponding to each row.
    et_labels : np.ndarray
        Log-space emergence timing labels of shape (N,). NaN = unavailable.
    """
    def __init__(
        self,
        X: np.ndarray,
        caseids: List[int],
        et_labels: np.ndarray,
    ) -> None:
        super().__init__()
        assert len(X) == len(caseids) == len(et_labels), \
            "X, caseids, and et_labels must have the same length"
        self.X = torch.from_numpy(X.astype(np.float32))  # (N, F)
        self.caseids = caseids
        self.et_labels = torch.from_numpy(et_labels.astype(np.float32))  # (N,)
    def __len__(self) -> int:
        return len(self.X)
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """
        Return a single patient sample.
        Returns
        ------
        dict with keys:
            x               : FloatTensor (n_features,)  — feature vector
            label_ioh_t0    : FloatTensor scalar          — not applicable → -1
            label_ioh_t30   : FloatTensor scalar          — not applicable → -1
            label_ioh_t60   : FloatTensor scalar          — not applicable → -1
            label_ioh_t180  : FloatTensor scalar          — not applicable → -1
            label_ioh_t300  : FloatTensor scalar          — not applicable → -1
            label_nh        : LongTensor scalar           — not applicable → -1
            label_et        : FloatTensor scalar          — log(emergence_sec) or NaN
            caseid          : int
            window_start    : int                         — not applicable → -1
            sqi_flag        : FloatTensor (1,)            — not applicable → 1.0
            modality_present: BoolTensor scalar           — always True for MLP
        """
        et = self.et_labels[idx]
        return {
            "x":                self.X[idx],                         # (F,)
            "label_ioh_t0":     torch.tensor(-1.0, dtype=torch.float32),
            "label_ioh_t30":    torch.tensor(-1.0, dtype=torch.float32),
            "label_ioh_t60":    torch.tensor(-1.0, dtype=torch.float32),
            "label_ioh_t180":   torch.tensor(-1.0, dtype=torch.float32),
            "label_ioh_t300":   torch.tensor(-1.0, dtype=torch.float32),
            "label_nh":         torch.tensor(-1,   dtype=torch.long),
            "label_et":         et,                                   # scalar float or NaN
            "caseid":           self.caseids[idx],
            "window_start":     -1,
            "sqi_flag":         torch.ones(1, dtype=torch.float32),
            "modality_present": torch.tensor(True,  dtype=torch.bool),
        }

# SECTION 10 — build_dataloaders()

In [0]:
def build_dataloaders(
    train_ds: Model5Dataset,
    val_ds: Model5Dataset,
    test_ds: Model5Dataset,
    config: Dict[str, Any],
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Construct DataLoader objects for train, validation, and test splits.
    Parameters
    ---------
    train_ds, val_ds, test_ds : Model5Dataset
        Dataset objects for each split.
    config : dict
        CONFIG dict.
    Returns
    ------
    train_loader, val_loader, test_loader : DataLoaders
    """
    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    LOGGER.info(
        "DataLoaders: train=%d batches, val=%d batches, test=%d batches",
        len(train_loader), len(val_loader), len(test_loader)
    )
    return train_loader, val_loader, test_loader

#  SECTION 11 — Model Architecture

In [0]:
class MLPBlock(nn.Module):
    """
    A single MLP block: Linear → BatchNorm1d → ReLU → Dropout.
    Parameters
    ---------
    in_dim : int
        Input dimension.
    out_dim : int
        Output dimension.
    dropout : float
        Dropout probability.
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through one MLP block.
        Parameters
        ---------
        x : torch.Tensor
            Input tensor of shape (batch, in_dim).
        Returns
        ------
        torch.Tensor
            Output tensor of shape (batch, out_dim).
        """
        return self.block(x)

In [0]:
class PatientContextMLP(nn.Module):
    """
    Patient Context MLP (Model 5).
    Architecture:
        Input(~25 dims) →
        Linear(in, 64) → BN → ReLU → Dropout(0.3) →
        Linear(64, 128) → BN → ReLU → Dropout(0.3) →
        Linear(128, 64) → BN → ReLU → Dropout(0.3) →
        Linear(64, 128) → [embedding output]
    The 128-dim output is a static patient embedding, broadcast to all
    timesteps in the fusion model.
    Additionally, a regression head predicts log(emergence_sec) for
    pretraining: Linear(128, 1).
    Parameters
    ---------
    input_dim : int
        Number of input features.
    hidden_dims : list of int
        Hidden layer sizes, matching CONFIG hidden_dims = [64, 128, 64].
    embedding_dim : int
        Final embedding dimension (128).
    dropout : float
        Dropout probability.
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int],
        embedding_dim: int = 128,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        assert len(hidden_dims) == 3, \
            "PatientContextMLP expects exactly 3 hidden layer dims: [64, 128, 64]"
        h1, h2, h3 = hidden_dims  # 64, 128, 64
        # Layer 1: input → 64
        self.layer1 = MLPBlock(input_dim, h1, dropout=dropout)
        # Layer 2: 64 → 128
        self.layer2 = MLPBlock(h1, h2, dropout=dropout)
        # Layer 3: 128 → 64
        self.layer3 = MLPBlock(h2, h3, dropout=dropout)
        # Final projection: 64 → 128 (the patient embedding)
        # No activation after this layer — the embedding is used as-is in fusion
        self.output_proj = nn.Linear(h3, embedding_dim)
        # Regression head for emergence timing pretraining
        # Predicts log(emergence_sec); trained with MSE loss
        self.et_regression_head = nn.Linear(embedding_dim, 1)
        # Weight initialisation: He/Kaiming for ReLU networks
        self._init_weights()
    def _init_weights(self) -> None:
        """
        Apply Kaiming uniform initialisation to all Linear layers
        and zero-initialise all BatchNorm biases.
        """
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through the Patient Context MLP.
        Parameters
        ---------
        x : torch.Tensor
            Input feature vector, shape (batch, input_dim).
        Returns
        ------
        embedding : torch.Tensor
            Patient embedding, shape (batch, 128).
        et_pred : torch.Tensor
            Log-space emergence time prediction, shape (batch, 1).
        """
        # Pass through the three hidden blocks
        h = self.layer1(x)    # (batch, 64)
        h = self.layer2(h)    # (batch, 128)
        h = self.layer3(h)    # (batch, 64)
        # Project to final embedding dimension
        embedding = self.output_proj(h)   # (batch, 128)
        # Emergence timing regression from the embedding
        et_pred = self.et_regression_head(embedding)  # (batch, 1)
        return embedding, et_pred


# SECTION 12 — Loss functions

In [0]:
def et_lognormal_loss(
    pred_log: torch.Tensor,
    target_log: torch.Tensor,
) -> torch.Tensor:
    """
    Log-normal MSE loss for emergence timing regression.
    Both pred_log and target_log are expected to be in log-space
    (i.e., log(emergence_sec)). NaN targets are masked out so cases
    without a valid ET label do not contribute to the loss.
    Parameters
    ---------
    pred_log : torch.Tensor
        Predicted log(emergence_sec), shape (batch, 1) or (batch,).
    target_log : torch.Tensor
        Ground-truth log(emergence_sec), shape (batch,). NaN = unavailable.
    Returns
    ------
    torch.Tensor
        Scalar loss (mean over valid samples). Returns 0 if no valid samples.
    """
    pred_log = pred_log.squeeze(-1)   # (batch,)
    # Mask out NaN targets (cases without valid ET label)
    valid_mask = ~torch.isnan(target_log)
    n_valid = valid_mask.sum().item()
    if n_valid == 0:
        # No valid ET labels in this batch — return zero loss (no gradient)
        return torch.tensor(0.0, device=pred_log.device, requires_grad=True)
    loss = nn.functional.mse_loss(
        pred_log[valid_mask],
        target_log[valid_mask],
        reduction="mean",
    )
    return loss


# SECTION 13 — compute_metrics()

In [0]:
def compute_metrics(
    preds_log: np.ndarray,
    targets_log: np.ndarray,
) -> Dict[str, float]:
    """
    Compute regression metrics for emergence timing prediction.
    Parameters
    ---------
    preds_log : np.ndarray
        Predicted log(emergence_sec), shape (N,).
    targets_log : np.ndarray
        Ground-truth log(emergence_sec), shape (N,). NaN masked.
    Returns
    ------
    dict
        MAE, RMSE, and MdAPE in original seconds space, plus MSE in log space.
    """
    valid = ~np.isnan(targets_log)
    if valid.sum() == 0:
        return {"mae_sec": np.nan, "rmse_sec": np.nan, "mdape_pct": np.nan, "mse_log": np.nan}
    p = preds_log[valid]
    t = targets_log[valid]
    # Log-space MSE
    mse_log = float(np.mean((p - t) ** 2))
    # Convert back to seconds for interpretable metrics
    p_sec = np.exp(p)
    t_sec = np.exp(t)
    mae_sec  = float(np.mean(np.abs(p_sec - t_sec)))
    rmse_sec = float(np.sqrt(np.mean((p_sec - t_sec) ** 2)))
    # Median Absolute Percentage Error (MdAPE)
    ape = np.abs((p_sec - t_sec) / (t_sec + 1e-6)) * 100.0
    mdape_pct = float(np.median(ape))
    return {
        "mse_log":   mse_log,
        "mae_sec":   mae_sec,
        "rmse_sec":  rmse_sec,
        "mdape_pct": mdape_pct,
    }


# SECTION 14 — train_one_epoch()

In [0]:
def train_one_epoch(
    model: PatientContextMLP,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    amp_scaler: GradScaler,
    device: torch.device,
    config: Dict[str, Any],
) -> float:
    """
    Run one training epoch with mixed-precision AMP and gradient clipping.
    Parameters
    ---------
    model : PatientContextMLP
    loader : DataLoader
        Training DataLoader.
    optimizer : torch.optim.Optimizer
    amp_scaler : GradScaler
        AMP gradient scaler.
    device : torch.device
    config : dict
    Returns
    ------
    float
        Mean training loss for this epoch.
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(loader, desc="  Train", leave=False):
        x = batch["x"].to(device, non_blocking=True)                  # (B, F)
        et_target = batch["label_et"].to(device, non_blocking=True)   # (B,)
        optimizer.zero_grad()
        # Mixed precision forward pass
        with autocast(device_type="cuda"):
            _embedding, et_pred = model(x)
            loss = et_lognormal_loss(et_pred, et_target)
        # Backward with AMP gradient scaling
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=config["grad_clip_norm"]
        )
        amp_scaler.step(optimizer)
        amp_scaler.update()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

# SECTION 15 — validate()

In [0]:
def validate(
    model: PatientContextMLP,
    loader: DataLoader,
    device: torch.device,
    config: Dict[str, Any],
) -> Tuple[float, Dict[str, float]]:
    """
    Run validation loop and compute metrics.
    Parameters
    ---------
    model : PatientContextMLP
    loader : DataLoader
        Validation DataLoader.
    device : torch.device
    config : dict
    Returns
    ------
    val_loss : float
        Mean validation loss.
    metrics : dict
        Dictionary of evaluation metrics.
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_preds: List[np.ndarray] = []
    all_targets: List[np.ndarray] = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Val  ", leave=False):
            x = batch["x"].to(device, non_blocking=True)
            et_target = batch["label_et"].to(device, non_blocking=True)
            with autocast(device_type="cuda"):
                _embedding, et_pred = model(x)
                loss = et_lognormal_loss(et_pred, et_target)
            total_loss += loss.item()
            n_batches += 1
            all_preds.append(et_pred.squeeze(-1).cpu().numpy())
            all_targets.append(et_target.cpu().numpy())
    preds_np   = np.concatenate(all_preds,   axis=0)
    targets_np = np.concatenate(all_targets, axis=0)
    metrics = compute_metrics(preds_np, targets_np)
    return total_loss / max(n_batches, 1), metrics

# SECTION 16 — EarlyStopping

In [0]:
class EarlyStopping:
    """
    Early stopping with patience-based tracking and best-model checkpointing.
    Parameters
    ---------
    patience : int
        Number of epochs without improvement before stopping.
    checkpoint_path : str
        Path to save the best model state dict.
    """
    def __init__(self, patience: int = 10, checkpoint_path: str = "best.pt") -> None:
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_loss: float = math.inf
        self.counter: int = 0
        self.should_stop: bool = False
    def step(self, val_loss: float, model: nn.Module) -> None:
        """
        Update early stopping state based on current validation loss.
        Parameters
        ---------
        val_loss : float
            Validation loss for the current epoch.
        model : nn.Module
            Model to checkpoint if val_loss improved.
        """
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            LOGGER.debug("EarlyStopping: new best val loss %.5f → checkpoint saved", val_loss)
        else:
            self.counter += 1
            LOGGER.debug(
                "EarlyStopping: no improvement (%d / %d)", self.counter, self.patience
            )
            if self.counter >= self.patience:
                self.should_stop = True
                LOGGER.info(
                    "EarlyStopping triggered after %d epochs without improvement", self.patience
                )

# SECTION 17 — train_model()

In [0]:
def train_model(
    model: PatientContextMLP,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    config: Dict[str, Any],
) -> PatientContextMLP:
    """
    Full training loop: runs up to max_epochs, with early stopping and AMP.
    Parameters
    ---------
    model : PatientContextMLP
    train_loader, val_loader : DataLoader
    device : torch.device
    config : dict
    Returns
    ------
    PatientContextMLP
        Model loaded with best checkpoint weights.
    """
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    # Cosine annealing LR scheduler — reduces LR smoothly over training
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["max_epochs"], eta_min=config["learning_rate"] * 0.01
    )
    amp_scaler = GradScaler(device_type="cuda")
    early_stopping = EarlyStopping(
        patience=config["early_stopping_patience"],
        checkpoint_path=config["checkpoint_path"],
    )
    LOGGER.info(
        "Starting training — max_epochs=%d, patience=%d",
        config["max_epochs"], config["early_stopping_patience"]
    )
    for epoch in range(1, config["max_epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, amp_scaler, device, config)
        val_loss, val_metrics = validate(model, val_loader, device, config)
        scheduler.step()
        LOGGER.info(
            "Epoch %03d | train_loss=%.5f | val_loss=%.5f | "
            "mae_sec=%.1f | rmse_sec=%.1f | mdape_pct=%.1f%%",
            epoch, train_loss, val_loss,
            val_metrics.get("mae_sec", float("nan")),
            val_metrics.get("rmse_sec", float("nan")),
            val_metrics.get("mdape_pct", float("nan")),
        )
        early_stopping.step(val_loss, model)
        if early_stopping.should_stop:
            LOGGER.info("Early stopping at epoch %d", epoch)
            break
    # Reload best checkpoint
    LOGGER.info("Loading best checkpoint from %s", config["checkpoint_path"])
    model.load_state_dict(torch.load(config["checkpoint_path"], map_location=device, weights_only=True))
    return model

# SECTION 18 — evaluate_test()


In [0]:
def evaluate_test(
    model: PatientContextMLP,
    test_loader: DataLoader,
    device: torch.device,
    config: Dict[str, Any],
) -> None:
    """
    Evaluate the trained model on the held-out test set and log full metrics.
    Parameters
    ---------
    model : PatientContextMLP
    test_loader : DataLoader
    device : torch.device
    config : dict
    """
    LOGGER.info("─── Test Evaluation ───────────────────────────────────────")
    test_loss, test_metrics = validate(model, test_loader, device, config)
    LOGGER.info("Test loss (log-normal MSE): %.5f", test_loss)
    for k, v in test_metrics.items():
        LOGGER.info("  %s: %.4f", k, v)
    # Also print to stdout for immediate visibility
    print("\n════ Test Set Metrics (Model 5 — Patient Context MLP) ════")
    print(f"  Log-space MSE loss : {test_loss:.5f}")
    print(f"  MSE (log space)    : {test_metrics.get('mse_log',   float('nan')):.5f}")
    print(f"  MAE (seconds)      : {test_metrics.get('mae_sec',   float('nan')):.2f} s")
    print(f"  RMSE (seconds)     : {test_metrics.get('rmse_sec',  float('nan')):.2f} s")
    print(f"  MdAPE              : {test_metrics.get('mdape_pct', float('nan')):.2f} %")
    print("═══════════════════════════════════════════════════════════\n")

# SECTION 19 — explain_with_shap()

In [0]:
def explain_with_shap(
    model: PatientContextMLP,
    test_loader: DataLoader,
    feature_names: List[str],
    device: torch.device,
    config: Dict[str, Any],
) -> None:
    """
    Compute SHAP values for the emergence timing regression head and save plot.
    Uses shap.DeepExplainer on the full model pipeline.
    Samples a background set from the test loader for efficiency.
    Parameters
    ---------
    model : PatientContextMLP
    test_loader : DataLoader
    feature_names : list of str
    device : torch.device
    config : dict
    """
    LOGGER.info("Computing SHAP values for Model 5 …")
    model.eval()
    # Collect all test inputs
    all_x: List[torch.Tensor] = []
    for batch in test_loader:
        all_x.append(batch["x"])
    X_all = torch.cat(all_x, dim=0).to(device)   # (N_test, F)
    # Use first 100 samples as SHAP background (DeepExplainer requirement)
    n_background = min(100, len(X_all))
    background = X_all[:n_background]
    # Wrap model to expose only the scalar ET prediction (SHAP needs scalar output)
    class _ETWrapper(nn.Module):
        def __init__(self, base: PatientContextMLP) -> None:
            super().__init__()
            self.base = base
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            _emb, et = self.base(x)
            return et  # (B, 1)
    wrapped = _ETWrapper(model)
    try:
        explainer = shap.DeepExplainer(wrapped, background)
        # Explain a subset for speed
        n_explain = min(200, len(X_all))
        shap_values = explainer.shap_values(X_all[:n_explain])
        # shap_values is a list (one element per output); we have one output
        sv = shap_values[0] if isinstance(shap_values, list) else shap_values
        sv = np.array(sv).squeeze()  # (n_explain, F)
        # Plot summary
        plt.figure(figsize=(12, 8))
        shap.summary_plot(
            sv,
            X_all[:n_explain].cpu().numpy(),
            feature_names=feature_names,
            show=False,
            max_display=20,
        )
        plt.title("SHAP Feature Importance — Model 5 (Emergence Timing)")
        plt.tight_layout()
        plt.savefig(config["shap_output"], dpi=150, bbox_inches="tight")
        plt.close()
        LOGGER.info("SHAP plot saved to %s", config["shap_output"])
    except Exception as exc:
        LOGGER.warning("SHAP computation failed: %s — skipping plot", exc)

# SECTION 20 — plot_predictions()

In [0]:
def plot_predictions(
    model: PatientContextMLP,
    test_loader: DataLoader,
    device: torch.device,
    config: Dict[str, Any],
    n_cases: int = 5,
) -> None:
    """
    Generate predicted vs. actual emergence timing scatter plot and save figure.
    Parameters
    ---------
    model : PatientContextMLP
    test_loader : DataLoader
    device : torch.device
    config : dict
    n_cases : int
        Minimum number of highlighted example cases to annotate (unused for MLP;
        all valid predictions are plotted as a scatter).
    """
    LOGGER.info("Generating prediction plots for Model 5 …")
    model.eval()
    all_preds_log:   List[np.ndarray] = []
    all_targets_log: List[np.ndarray] = []
    with torch.no_grad():
        for batch in test_loader:
            x  = batch["x"].to(device, non_blocking=True)
            et = batch["label_et"].cpu().numpy()
            with autocast(device_type="cuda"):
                _emb, et_pred = model(x)
            all_preds_log.append(et_pred.squeeze(-1).cpu().numpy())
            all_targets_log.append(et)
    preds_log   = np.concatenate(all_preds_log,   axis=0)
    targets_log = np.concatenate(all_targets_log, axis=0)
    # Filter to valid (non-NaN) targets
    valid = ~np.isnan(targets_log)
    p = np.exp(preds_log[valid])    # back to seconds
    t = np.exp(targets_log[valid])
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Model 5 — Patient Context MLP\nEmergence Timing Predictions (Test Set)", 
fontsize=13)
    # 
── Left: scatter predicted vs actual ─────────────────────────────────────
    ax = axes[0]
    ax.scatter(t, p, alpha=0.3, s=10, color="steelblue", label="Predictions")
    lim_max = max(t.max(), p.max()) * 1.05
    ax.plot([0, lim_max], [0, lim_max], "r--", linewidth=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual emergence time (s)")
    ax.set_ylabel("Predicted emergence time (s)")
    ax.set_title("Predicted vs Actual")
    ax.legend(fontsize=8)
    ax.set_xlim(0, lim_max)
    ax.set_ylim(0, lim_max)
    # 
── Right: residual distribution ──────────────────────────────────────────
    ax = axes[1]
    residuals_sec = p - t
    ax.hist(residuals_sec, bins=50, color="coral", edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=1.5, linestyle="--")
    ax.set_xlabel("Residual (predicted − actual, seconds)")
    ax.set_ylabel("Count")
    ax.set_title("Residual Distribution")
    plt.tight_layout()
    plt.savefig(config["pred_output"], dpi=150, bbox_inches="tight")
    plt.close()
    LOGGER.info("Prediction plot saved to %s", config["pred_output"])

In [0]:
def main() -> None:
    """
    Full pipeline for Model 5 — Patient Context MLP.
    Execution order:
    1.  Load clinical metadata + lab data
    2.  Merge, apply eligibility filter, remove outliers
    3.  Build emergence timing labels
    4.  Split cases (patient-level)
    5.  Build feature matrices (fit on train, apply to val/test)
    6.  Construct Datasets and DataLoaders
    7.  Instantiate PatientContextMLP
    8.  Train with early stopping
    9.  Evaluate on test set
    10. SHAP explanation
    11. Prediction visualisation
    """
    LOGGER.info("══════════════════════════════════════════════════════════")
    LOGGER.info("Model 5 — Patient Context MLP — pipeline start")
    LOGGER.info("══════════════════════════════════════════════════════════")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    LOGGER.info("Device: %s", device)
    # ── 1. Load data from VitalDB ─────────────────────────────────────────────
    df_clin = load_clinical_metadata(CONFIG)
    df_labs = load_lab_data_wide(CONFIG)
    # Merge clinical + labs on caseid (left join — all cases kept)
    df = pd.merge(df_clin, df_labs, on="caseid", how="left")
    LOGGER.info("Merged DataFrame: %d rows", len(df))
    # ── 2. Eligibility filter + outlier removal ───────────────────────────────
    df = apply_eligibility_filter(df, CONFIG)
    df = remove_outliers(df)
    # ── 3. Build emergence timing labels ─────────────────────────────────────
    df = build_et_labels(df, CONFIG)
    # ── 4. Split cases (patient-level) ────────────────────────────────────────
    all_caseids = df["caseid"].tolist()
    train_ids, val_ids, test_ids = split_cases(
        all_caseids,
        val_frac=CONFIG["val_frac"],
        test_frac=CONFIG["test_frac"],
        seed=CONFIG["random_seed"],
    )
    df_train = df[df["caseid"].isin(train_ids)].reset_index(drop=True)
    df_val   = df[df["caseid"].isin(val_ids)].reset_index(drop=True)
    df_test  = df[df["caseid"].isin(test_ids)].reset_index(drop=True)
    LOGGER.info(
        "Data sizes — train: %d, val: %d, test: %d",
        len(df_train), len(df_val), len(df_test)
    )
    # ── 5. Build feature matrices ─────────────────────────────────────────────
    # Fit scalers/imputers on training data ONLY (no data leakage)
    X_train, feature_names, prep_artifacts = build_feature_matrix(
        df_train, CONFIG, fit_mode=True, artifacts=None
    )
    X_val, _, _ = build_feature_matrix(
        df_val, CONFIG, fit_mode=False, artifacts=prep_artifacts
    )
    X_test, _, _ = build_feature_matrix(
        df_test, CONFIG, fit_mode=False, artifacts=prep_artifacts
    )
    LOGGER.info(
        "Feature matrix shapes — train: %s, val: %s, test: %s",
        X_train.shape, X_val.shape, X_test.shape
    )
    LOGGER.info("Feature names (%d): %s", len(feature_names), feature_names)
    # Save preprocessing artifacts for inference / fusion model compatibility
    artifacts_path = "model_5_prep_artifacts.pkl"
    with open(artifacts_path, "wb") as f:
        pickle.dump({"prep_artifacts": prep_artifacts, "feature_names": feature_names}, f)
    LOGGER.info("Preprocessing artifacts saved to %s", artifacts_path)
    # ─ 6. ET label arrays ────────────────────────────────────────────────────
    def _extract_et(df_split: pd.DataFrame) -> np.ndarray:
        # Return log_emergence_sec as float array; NaN preserved
        return df_split["log_emergence_sec"].values.astype(np.float32)
    et_train = _extract_et(df_train)
    et_val   = _extract_et(df_val)
    et_test  = _extract_et(df_test)
    n_valid_train = (~np.isnan(et_train)).sum()
    LOGGER.info(
        "ET labels with valid values — train: %d / %d, val: %d / %d, test: %d / %d",
        n_valid_train, len(et_train),
        (~np.isnan(et_val)).sum(), len(et_val),
        (~np.isnan(et_test)).sum(), len(et_test),
    )
    # ── 7. Datasets and DataLoaders ───────────────────────────────────────────
    train_ds = Model5Dataset(X_train, df_train["caseid"].tolist(), et_train)
    val_ds   = Model5Dataset(X_val,   df_val["caseid"].tolist(),   et_val)
    test_ds  = Model5Dataset(X_test,  df_test["caseid"].tolist(),  et_test)
    train_loader, val_loader, test_loader = build_dataloaders(
        train_ds, val_ds, test_ds, CONFIG
    )
    # ── 8. Instantiate model ──────────────────────────────────────────────────
    input_dim = X_train.shape[1]
    LOGGER.info("Instantiating PatientContextMLP with input_dim=%d", input_dim)
    model = PatientContextMLP(
        input_dim=input_dim,
        hidden_dims=CONFIG["hidden_dims"],
        embedding_dim=CONFIG["embedding_dim"],
        dropout=CONFIG["dropout"],
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    LOGGER.info("Model parameters: %d", n_params)
    # ── 9. Train ──────────────────────────────────────────────────────────────
    model = train_model(model, train_loader, val_loader, device, CONFIG)
    # ── 10. Test evaluation ────────────────────────────────────────────────────
    evaluate_test(model, test_loader, device, CONFIG)
    # ── 11. SHAP ──────────────────────────────────────────────────────────────
    explain_with_shap(model, test_loader, feature_names, device, CONFIG)
    # ── 12. Prediction visualisation ──────────────────────────────────────────
    plot_predictions(model, test_loader, device, CONFIG, n_cases=5)
    LOGGER.info("══════════════════════════════════════════════════════════")
    LOGGER.info("Model 5 pipeline complete.")
    LOGGER.info("══════════════════════════════════════════════════════════")
    # Explicit cleanup
    del train_ds, val_ds, test_ds
    del X_train, X_val, X_test
    gc.collect()

In [0]:
if __name__ == "__main__":
    main()